# 🎯 LoRA and PEFT

**Parameter-Efficient Fine-Tuning for large models**

---

## 📋 Overview

**What you'll learn:**
- What is LoRA (Low-Rank Adaptation)
- PEFT techniques comparison
- Implementing LoRA with HuggingFace
- Merging and deploying LoRA adapters
- Cost and efficiency benefits

**Time estimate:** ⏱️ 60 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
# Install required packages
# !pip install transformers peft accelerate bitsandbytes datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import numpy as np

print("✅ Setup complete")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

## 🤔 Why LoRA?

### The Problem with Full Fine-Tuning:

```
LLaMA-7B model:
  Parameters: 7 billion
  Memory for training: ~28 GB (fp32)
  Training time: Hours to days
  Cost: $$$$
```

### LoRA Solution:

```
LoRA for LLaMA-7B:
  Trainable parameters: ~4 million (0.06%!)
  Memory: ~8 GB
  Training time: 10x faster
  Cost: 90% cheaper
  Performance: Nearly identical!
```

### How LoRA Works:

**Traditional fine-tuning:**
```python
W' = W + ΔW  # Update all weights
```

**LoRA:**
```python
W' = W + BA  # W frozen, only train B and A (low-rank matrices)
# B: d × r, A: r × k  (r << d, k)
# Trainable params: r(d + k) instead of d × k
```

### Key Benefits:

- 📉 **Fewer parameters**: 0.01% - 1% of original
- 💰 **Lower cost**: 90% cheaper training
- ⚡ **Faster**: 3-10x speedup
- 💾 **Smaller**: Adapters are MB not GB
- 🔄 **Swappable**: Multiple tasks, one base model

## 🆚 PEFT Techniques Comparison

In [ ]:
import pandas as pd

# Compare PEFT methods
peft_comparison = pd.DataFrame([
    {
        'Method': 'Full Fine-Tuning',
        'Trainable %': '100%',
        'Memory': 'Very High',
        'Speed': '1x',
        'Performance': '100%',
        'Adapter Size': 'Full model',
    },
    {
        'Method': 'LoRA',
        'Trainable %': '0.1-1%',
        'Memory': 'Low',
        'Speed': '3-5x',
        'Performance': '95-99%',
        'Adapter Size': '~10 MB',
    },
    {
        'Method': 'Prefix Tuning',
        'Trainable %': '0.01-0.1%',
        'Memory': 'Very Low',
        'Speed': '5-10x',
        'Performance': '85-95%',
        'Adapter Size': '~1 MB',
    },
    {
        'Method': 'Adapter Layers',
        'Trainable %': '1-5%',
        'Memory': 'Medium',
        'Speed': '2-3x',
        'Performance': '90-95%',
        'Adapter Size': '~50 MB',
    },
])

print("🆚 PEFT Methods Comparison\n")
print(peft_comparison.to_string(index=False))
print("\n💡 LoRA offers the best balance of efficiency and performance")

## 🏗️ Implementing LoRA

In [ ]:
# Load base model
model_name = "gpt2"  # Small model for demo

print(f"📥 Loading base model: {model_name}")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Count parameters before LoRA
total_params_before = sum(p.numel() for p in model.parameters())
trainable_params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Base Model Stats:")
print(f"   Total parameters: {total_params_before:,}")
print(f"   Trainable: {trainable_params_before:,}")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=8,  # Rank of the low-rank matrices
    lora_alpha=32,  # Scaling factor
    target_modules=["c_attn"],  # Which modules to apply LoRA to
    lora_dropout=0.1,  # Dropout for regularization
    bias="none",  # Don't train bias
    task_type=TaskType.CAUSAL_LM  # Task type
)

print("⚙️  LoRA Configuration:")
print(f"   Rank (r): {lora_config.r}")
print(f"   Alpha: {lora_config.lora_alpha}")
print(f"   Target modules: {lora_config.target_modules}")
print(f"   Dropout: {lora_config.lora_dropout}")

# Apply LoRA
model = get_peft_model(model, lora_config)

# Count parameters after LoRA
total_params_after = sum(p.numel() for p in model.parameters())
trainable_params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 With LoRA:")
print(f"   Total parameters: {total_params_after:,}")
print(f"   Trainable: {trainable_params_after:,}")
print(f"   Trainable %: {trainable_params_after/total_params_after*100:.3f}%")
print(f"   Reduction: {100 - (trainable_params_after/total_params_before*100):.1f}%")

model.print_trainable_parameters()

## 📚 Prepare Training Data

In [ ]:
# Create sample dataset
training_examples = [
    "Python is a high-level programming language known for simplicity.",
    "Machine learning models learn patterns from data.",
    "Natural language processing helps computers understand text.",
    "Deep learning uses neural networks with multiple layers.",
    "Data science combines statistics, programming, and domain knowledge.",
] * 10  # Repeat for more data

# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples,
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

# Create dataset
dataset = Dataset.from_dict({"text": training_examples})

def tokenize_dataset(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_dataset, batched=True, remove_columns=["text"])

print(f"📊 Dataset:")
print(f"   Examples: {len(tokenized_dataset)}")
print(f"   Features: {tokenized_dataset.column_names}")

## 🎯 Train with LoRA

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",  # Disable wandb/tensorboard
)

print("⚙️  Training Configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("\n🏋️  Starting training...")
print("💡 In production, this would train for several hours")
print("   Uncomment the line below to actually train:\n")

# Uncomment to train:
# trainer.train()

print("# trainer.train()")
print("\n✅ Training complete (simulated)")

## 💾 Save and Load LoRA Adapters

In [ ]:
# Save LoRA adapters
lora_path = "./my_lora_adapter"

print(f"💾 Saving LoRA adapter to {lora_path}")
# model.save_pretrained(lora_path)

print("\n📁 Saved files:")
print("   adapter_config.json  # LoRA configuration")
print("   adapter_model.bin    # Trained weights (~10 MB!)")

print("\n💡 Compare sizes:")
print("   Full model: ~500 MB")
print("   LoRA adapter: ~10 MB (50x smaller!)")

In [ ]:
# Load LoRA adapter
from peft import PeftModel

def load_lora_model(base_model_name: str, lora_path: str):
    """Load base model + LoRA adapter."""
    
    print(f"📥 Loading base model: {base_model_name}")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    
    print(f"📥 Loading LoRA adapter: {lora_path}")
    model_with_lora = PeftModel.from_pretrained(base_model, lora_path)
    
    return model_with_lora

print("💡 Loading example:\n")
print("""
# Load model with adapter
model = load_lora_model("gpt2", "./my_lora_adapter")

# Use for inference
output = model.generate(...)
""")

## 🔄 Merging and Deployment

In [ ]:
# Option 1: Use with adapter (dynamic)
print("📌 Deployment Options:\n")

print("1️⃣  Dynamic Adapter Loading (Recommended):")
print("""
# Load base once, swap adapters
base = AutoModelForCausalLM.from_pretrained("gpt2")

# Customer support
support_model = PeftModel.from_pretrained(base, "./support_adapter")

# Sales
sales_model = PeftModel.from_pretrained(base, "./sales_adapter")

# Multiple tasks, one base model!
""")

print("\n2️⃣  Merge Adapter into Base (Static):")
print("""
# Merge for faster inference
model = PeftModel.from_pretrained(base_model, lora_path)
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained("./merged_model")

# Use like normal model
model = AutoModelForCausalLM.from_pretrained("./merged_model")
""")

print("\n⚡ Performance Comparison:")
print("   Dynamic adapter: Slightly slower, swappable")
print("   Merged model: Fastest, single-task")

## 📊 LoRA Hyperparameters

In [ ]:
# LoRA hyperparameter guide
print("⚙️  LoRA Hyperparameter Guide\n")
print("="*60)

print("\n1. Rank (r):")
print("   Controls capacity of adaptation")
print("   • r=1-4:   Very simple tasks, fastest")
print("   • r=8:     Default, good balance")
print("   • r=16-32: Complex tasks, more capacity")
print("   • r=64+:   Overkill, approaching full fine-tuning")

print("\n2. Alpha (lora_alpha):")
print("   Scaling factor, typically alpha = 2 × r")
print("   • r=8  → alpha=16")
print("   • r=16 → alpha=32")

print("\n3. Target Modules:")
print("   Which layers to apply LoRA")
print("   • Attention only: [\"q_proj\", \"v_proj\"]  # Cheapest")
print("   • All attention: [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"]")
print("   • Attention + MLP: [...attention..., \"gate_proj\", \"up_proj\", \"down_proj\"]")

print("\n4. Dropout:")
print("   Regularization to prevent overfitting")
print("   • 0.0:  No dropout, small datasets")
print("   • 0.1:  Default")
print("   • 0.2+: High regularization, risk of underfitting")

print("\n💡 Recommended Starting Point:")
print("""
LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
""")

## ✅ Summary

### What is LoRA?

**Low-Rank Adaptation:**
- Freeze base model weights
- Add small trainable matrices (rank r)
- Train only 0.1-1% of parameters
- 90% cost reduction vs full fine-tuning

### Key Benefits:

```python
# Training Efficiency
Full fine-tuning:  100% params, 28 GB, $100
LoRA:              0.1% params,  8 GB,  $10

# Deployment
Full model:        500 MB per task
LoRA adapter:       10 MB per task (50x smaller!)
```

### When to Use LoRA:

✅ **Perfect for:**
- Large models (7B+ parameters)
- Limited compute budget
- Multiple task-specific models
- Fast iteration/experimentation

❌ **Consider full fine-tuning when:**
- Small models (< 1B params)
- Drastically changing model behavior
- Unlimited compute available

### Typical LoRA Configuration:

```python
# Balanced (recommended)
LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1
)

# Maximum efficiency
LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj"],  # Attention queries only
    lora_dropout=0.0
)

# Maximum performance
LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1
)
```

### Training Workflow:

```python
# 1. Load base model
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b")

# 2. Apply LoRA
lora_config = LoraConfig(r=8, lora_alpha=16)
model = get_peft_model(model, lora_config)

# 3. Train (only LoRA weights)
trainer = Trainer(model=model, ...)
trainer.train()

# 4. Save adapter (~10 MB)
model.save_pretrained("./my_adapter")

# 5. Load and use
model = PeftModel.from_pretrained(base_model, "./my_adapter")
```

### Multi-Task Deployment:

```python
# One base model, multiple adapters
base = AutoModelForCausalLM.from_pretrained("llama-2-7b")

# Task 1: Customer support
support_model = PeftModel.from_pretrained(base, "./support_lora")

# Task 2: Code generation  
code_model = PeftModel.from_pretrained(base, "./code_lora")

# Task 3: Translation
translate_model = PeftModel.from_pretrained(base, "./translate_lora")

# Storage: 7 GB (base) + 30 MB (3 adapters)
# vs 21 GB for 3 full models!
```

### Performance Comparison:

| Metric | Full FT | LoRA (r=8) | LoRA (r=16) |
|--------|---------|------------|-------------|
| **Trainable params** | 7B | 4M | 8M |
| **Memory** | 28 GB | 8 GB | 10 GB |
| **Training time** | 10h | 2h | 3h |
| **Accuracy** | 100% | 97% | 99% |
| **Adapter size** | 7 GB | 10 MB | 20 MB |

### Best Practices:

1. **Start with r=8**
   - Good default for most tasks
   - Increase if underfitting
   - Decrease if overfitting

2. **Target attention layers first**
   - Most effective for adaptation
   - Add MLP layers if needed

3. **Monitor training carefully**
   - LoRA can overfit quickly
   - Use validation set
   - Early stopping recommended

4. **Quantization + LoRA (QLoRA)**
   ```python
   # Load in 4-bit
   model = AutoModelForCausalLM.from_pretrained(
       "meta-llama/Llama-2-7b",
       load_in_4bit=True,
       bnb_4bit_compute_dtype=torch.float16
   )
   # Even lower memory: ~4 GB!
   ```

### Next: `06_fine_tuning/05_evaluation.ipynb`